# Ridership Analysis

The goal of this notebook is to analyze the ridership data for each ace bus, and identify trends and patterns over time.

Goals:

- Load and clean the ridership data
- Visualize ridership trends over time
- Identify peak ridership periods
- Compare ridership across different routes


## Setup and Analysis


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "..")))

import pandas as pd  # data wrangler library, dataframes are used to display and manipulate data
import seaborn as sns  # data graphing library, built on top of matplotlib
import matplotlib.pyplot as plt  # graphing library, used for titles and customization
import urllib.parse  # library to parse URLs for querying
import folium  # library to create interactive maps
import folium.plugins as plugins  # plugins for folium, used for clustering map points
import geopandas as gpd  # library to handle geospatial data
from notebooks.utils.utils import SOQL_Querying  # soql query object for help
from scipy import stats

## Initial Ridership Data Exploration


In [ ]:
qo = SOQL_Querying()

In [ ]:
ridership_columns = qo.pipeline(
    api="ridership",
    query="""
    select * limit 1
    """,
)
ridership_columns.info()

In [ ]:
routes = qo.pipeline(
    api="routes",
    query="""
    select * where program = "ACE"
    """,
)
routes.info()

In [ ]:
routes.head()

In [ ]:
routes.tail()

In [ ]:
routes["implementation_date"] = pd.to_datetime(routes["implementation_date"])
routes.head()

In [ ]:
ace_route_list = routes[routes["implementation_date"].dt.year == 2024]["route"].tolist()  # type: ignore
formatted_routes = str(tuple(ace_route_list))

In [ ]:
all_2023_data = []
current_max_date = "2024-01-01T00:00:00"
floor_date = "2023-01-01T00:00:00"

while True:
    query = f"""
    SELECT bus_route, transit_timestamp, ridership, transfers
    WHERE bus_route IN {formatted_routes}
    AND transit_timestamp < '{current_max_date}'
    AND transit_timestamp >= '{floor_date}'
    ORDER BY transit_timestamp DESC
    """

    chunk = qo.pipeline(api="ridership", query=query)

    if chunk.empty:
        break

    all_2023_data.append(chunk)

    chunk["transit_timestamp"] = pd.to_datetime(chunk["transit_timestamp"])
    oldest_in_chunk = chunk["transit_timestamp"].min()
    current_max_date = oldest_in_chunk.strftime("%Y-%m-%dT%H:%M:%S")

    print(f"Downloaded data down to: {current_max_date}")

raw_2023_complete = pd.concat(all_2023_data, ignore_index=True).sort_values(
    "transit_timestamp"
)
raw_2023_complete.info()

In [ ]:
all_2025_data = []
current_max_date = "2026-01-01T00:00:00"
floor_date = "2025-01-01T00:00:00"

while True:
    query = f"""
    SELECT bus_route, transit_timestamp, ridership, transfers
    WHERE bus_route IN {formatted_routes}
    AND transit_timestamp < '{current_max_date}'
    AND transit_timestamp >= '{floor_date}'
    ORDER BY transit_timestamp DESC
    LIMIT 1000
    """

    chunk = qo.pipeline(api="ridership-2025", query=query)

    if chunk.empty:
        print("Reached the beginning of 2025 or no more data found.")
        break

    all_2025_data.append(chunk)

    chunk["transit_timestamp"] = pd.to_datetime(chunk["transit_timestamp"])
    oldest_in_chunk = chunk["transit_timestamp"].min()
    current_max_date = oldest_in_chunk.strftime("%Y-%m-%dT%H:%M:%S")

    print(f"Downloaded data down to: {current_max_date}")

raw_2025_complete = pd.concat(all_2025_data, ignore_index=True).sort_values(
    "transit_timestamp"
)
raw_2025_complete.info()